In [7]:
import os
from pathlib import Path
import logging
import shutil
from datetime import datetime
import pandas as pd
from pyspark.sql import functions as F

import teehr
from teehr.evaluation.evaluation import RemoteReadOnlyEvaluation
from teehr.evaluation.spark_session_utils import create_spark_session
from teehr.fetching.nwm.nwm_points import nwm_to_parquet
from teehr.fetching.utils import create_periods_based_on_chunksize, get_period_start_end_times
from teehr.fetching.utils import (
    format_nwm_variable_name,
    format_nwm_configuration_metadata
)
from teehr.utils.utils import remove_dir_if_exists
from teehr.fetching.const import NWM_VARIABLE_MAPPER

import dask
dask.config.set({"distributed.dashboard.link": "{JUPYTERHUB_SERVICE_PREFIX}proxy/{port}/status"})

logger = logging.getLogger()

In [8]:
LOCATION_ID_PREFIX = "nwm30"
OCONUS_STATE_NAMES = ['Alaska']
nwm_version = "nwm30"

nwm_configuration = "medium_range_blend_alaska"
output_type = "channel_rt"
variable_name = "streamflow"

- NWM v3.0: 2023-09-19 - present

#### One time: Get the NWM v3.0 location IDs to fetch and save to csv

In [2]:
# %%time
# spark = create_spark_session()  # For read-only
# ev = teehr.RemoteReadOnlyEvaluation(spark=spark)

In [3]:
# %%time
# # Need to filter for OCONUS
# included_states = ", ".join(f"'{s}'" for s in OCONUS_STATE_NAMES)
# filtered_crosswalks_sdf = ev.location_crosswalks.add_attributes(
#     attr_list=["state_name"]
# ).filter(
#     filters=[
#         {
#             "column": "secondary_location_id",
#             "operator": "like",
#             "value": f"{LOCATION_ID_PREFIX}-%"
#         },
#         f"state_name IN ({included_states})"
#     ]
# ).to_sdf()
# stripped_ids = [
#     row[0].split("-")[1]
#     for row in filtered_crosswalks_sdf.select("secondary_location_id").collect()
# ]

In [4]:
# df = pd.DataFrame({"nwm_id": stripped_ids})
# df.to_csv("/data/data_processing/backfill-nwm/nwm_puertorico_ids.csv", index=False)

In [5]:
# ev.spark.stop()

#### Create a local Evaluation to store the cached parquet files

In [9]:
%%time
dir_path =  "/data/data_processing/backfill-nwm/nwm-alaska-teehr-warehouse"

spark = create_spark_session()
ev = teehr.LocalReadWriteEvaluation(
    spark=spark,
    dir_path=dir_path,
    create_dir=True
)

INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:✅ Spark local configuration successful!
INFO:teehr.evaluation.spark_session_utils:Setting Hadoop's default AWS credentials provider and AWS region
INFO:teehr.evaluation.spark_session_utils:🔑 Using AWS session token from boto3
INFO:teehr.evaluation.spark_session_utils:Configuring Iceberg catalogs...
INFO:teehr.evaluation.spark_session_utils:⚙️ All settings applied. Creating Spark session...
INFO:teehr.evaluation.spark_session_utils:🎉 Spark session created successfully!
INFO:teehr.evaluation.evaluation:Using provided Spark session.
INFO:teehr.evaluation.evaluation:Using existing directory /data/data_processing/backfill-nwm/nwm-alaska-teehr-warehouse.
INFO:teehr.utilities.apply_migrations:No new schema versions to apply to local.teehr.
INFO:teehr.evaluation.evaluation:Active catalog set to local.


CPU times: user 131 ms, sys: 40.3 ms, total: 172 ms
Wall time: 30.1 s


Get the CONUS NWM IDs (these were pre-prepared from the warehouse using the above query)

In [10]:
df_nwm_ids = pd.read_csv("/data/data_processing/backfill-nwm/nwm_alaska_ids.csv")
stripped_ids = df_nwm_ids["nwm_id"].tolist()

In [11]:
ev_variable_name = format_nwm_variable_name(variable_name)
ev_config = format_nwm_configuration_metadata(
    nwm_config_name=nwm_configuration,
    nwm_version=nwm_version
)
nwm_cache_dir = Path(
    ev.cache_dir,
    "fetching",
    "nwm"
)
kerchunk_cache_dir = Path(
    ev.cache_dir,
    "fetching",
    "kerchunk"
)

INFO:teehr.fetching.utils:Getting schema variable name for streamflow.
INFO:teehr.fetching.utils:Formatting configuration name for medium_range_blend_alaska.


In [12]:
start_date = datetime(2023, 9, 19, 0)  # 2023-09-19 (start of nwm v3.0)
end_date = datetime(2026, 4, 22, 0)   # 2025-11-10 22:00 (the earliest non-null reference_time in secondary)
chunk_by = "week"  # Describes the chunk size between start/end date

In [13]:
# Start up a local Dask cluster
from dask.distributed import Client
client = Client(n_workers=12)  # only using 4 workers by default?
client

INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:44859
INFO:distributed.scheduler:  dashboard at:  /user/samlamont/backfill-alaska-medium-range-blend/proxy/8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:36627'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:40793'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:46651'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:36435'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:43023'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:41663'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:39723'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:43927'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:43033'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:40995'
INFO:dis

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/samlamont/backfill-alaska-medium-range-blend/proxy/8787/status,
Dashboard: /user/samlamont/backfill-alaska-medium-range-blend/proxy/8787/status,Workers: 12
Total threads: 24,Total memory: 124.44 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:44859,Workers: 0
Dashboard: /user/samlamont/backfill-alaska-medium-range-blend/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:38921,Total threads: 2
Dashboard: /user/samlamont/backfill-alaska-medium-range-blend/proxy/39991/status,Memory: 10.37 GiB
Nanny: tcp://127.0.0.1:36627,


INFO:distributed.core:Event loop was unresponsive in Nanny for 3.13s.  This is often caused by long-running GIL-holding functions or moving large chunks of data. This can cause timeouts and instability.
INFO:distributed.core:Event loop was unresponsive in Nanny for 3.13s.  This is often caused by long-running GIL-holding functions or moving large chunks of data. This can cause timeouts and instability.
INFO:distributed.core:Event loop was unresponsive in Nanny for 3.13s.  This is often caused by long-running GIL-holding functions or moving large chunks of data. This can cause timeouts and instability.
INFO:distributed.core:Event loop was unresponsive in Scheduler for 3.14s.  This is often caused by long-running GIL-holding functions or moving large chunks of data. This can cause timeouts and instability.
INFO:distributed.core:Event loop was unresponsive in Nanny for 3.14s.  This is often caused by long-running GIL-holding functions or moving large chunks of data. This can cause timeout

Note: `chunk_by` doesn't have much meaning here, unless we want to do something with the cached files each iteration. <br>
The actual processing is done in chunks governed by:
- `process_by_z_hour` - chunks by reference time
- `step_size` - if above is False, an integer number of timesteps

In [15]:
%%time
periods = create_periods_based_on_chunksize(
    start_date=start_date,
    end_date=end_date,
    chunk_by=chunk_by
)

for i, period in enumerate(periods):

    # if i <= 2:  # already processed this batch
    #     continue 
    
    if period is not None:
        dts = get_period_start_end_times(
            period=period,
            start_date=start_date,
            end_date=end_date
        )
    else:
        dts = {"start_dt": start_date, "end_dt": end_date}

    logger.info("Fetching NWM data and writing to cache")
    nwm_to_parquet(
        configuration=nwm_configuration,
        output_type=output_type,
        variable_name=variable_name,
        start_date=dts["start_dt"],
        end_date=dts["end_dt"],
        location_ids=stripped_ids,
        json_dir=kerchunk_cache_dir,
        output_parquet_dir=Path(
            nwm_cache_dir,
            ev_config["name"],
            ev_variable_name
        ),
        nwm_version=nwm_version,
        variable_mapper=NWM_VARIABLE_MAPPER,
        kerchunk_method="auto",
        process_by_z_hour=False,
        stepsize=1440,
        overwrite_output=True
    )    

INFO:root:Fetching NWM data and writing to cache
INFO:teehr.fetching.nwm.nwm_points:Fetching medium_range_blend_alaska data. Version: nwm30
INFO:teehr.fetching.utils:Limiting the start date to z-hour: 0.
INFO:teehr.fetching.utils:Limiting the end date to z-hour: 23.
INFO:root:Fetching NWM data and writing to cache
INFO:teehr.fetching.nwm.nwm_points:Fetching medium_range_blend_alaska data. Version: nwm30
INFO:teehr.fetching.utils:Limiting the start date to z-hour: 0.
INFO:teehr.fetching.utils:Limiting the end date to z-hour: 23.
INFO:root:Fetching NWM data and writing to cache
INFO:teehr.fetching.nwm.nwm_points:Fetching medium_range_blend_alaska data. Version: nwm30
INFO:teehr.fetching.utils:Limiting the start date to z-hour: 0.
INFO:teehr.fetching.utils:Limiting the end date to z-hour: 23.
INFO:root:Fetching NWM data and writing to cache
INFO:teehr.fetching.nwm.nwm_points:Fetching medium_range_blend_alaska data. Version: nwm30
INFO:teehr.fetching.utils:Limiting the start date to z-hour

CPU times: user 2h 25min 54s, sys: 5min 55s, total: 2h 31min 50s
Wall time: 5h 28min 33s


In [14]:
240 * 6

1440

In [24]:
432 * 3

1296

In [26]:
parquet_cache_dir = Path(
    ev.cache_dir,
    "fetching",
    "nwm",
    ev_config["name"],
    ev_variable_name    
)
parquet_cache_dir

PosixPath('/data/data_processing/backfill-nwm/nwm-puerto-rico-short-teehr-warehouse/local/cache/fetching/nwm/nwm30_short_range_puertorico/streamflow_hourly_inst')

In [27]:
cached_sdf = ev.spark.read.parquet(parquet_cache_dir.as_posix())

In [28]:
cached_sdf.show(n=6, truncate=False)

+-------------------+-------------------+-----+----------------------+----------------------------+---------+---------------+------+----------+----------+
|reference_time     |value_time         |value|variable_name         |configuration_name          |unit_name|location_id    |member|created_at|updated_at|
+-------------------+-------------------+-----+----------------------+----------------------------+---------+---------------+------+----------+----------+
|2025-04-28 06:00:00|2025-04-28 07:00:00|1.4  |streamflow_hourly_inst|nwm30_short_range_puertorico|m^3/s    |nwm30-800035037|NULL  |NULL      |NULL      |
|2025-04-28 06:00:00|2025-04-28 07:00:00|3.98 |streamflow_hourly_inst|nwm30_short_range_puertorico|m^3/s    |nwm30-800034965|NULL  |NULL      |NULL      |
|2025-04-28 06:00:00|2025-04-28 07:00:00|0.69 |streamflow_hourly_inst|nwm30_short_range_puertorico|m^3/s    |nwm30-800030234|NULL  |NULL      |NULL      |
|2025-04-28 06:00:00|2025-04-28 07:00:00|0.16 |streamflow_hourly_inst|

In [29]:
cached_sdf.select(F.min("reference_time")).show(), cached_sdf.select(F.max("reference_time")).show()

+-------------------+
|min(reference_time)|
+-------------------+
|2023-09-19 06:00:00|
+-------------------+

+-------------------+
|max(reference_time)|
+-------------------+
|2026-04-21 18:00:00|
+-------------------+



(None, None)

In [30]:
cached_sdf.select("location_id").distinct().count()

83

There are 83 locations for Puerto Rico